In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load and Prepare
column_names = ['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave_points_worst', 'symmetry_worst', 'fractal_dimension_worst']
df = pd.read_csv('data/wdbc.data', names=column_names, header=None)
df.drop('id', axis=1, inplace=True)
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

X = df.drop('diagnosis', axis=1)
y = df['diagnosis']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [2]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# Hyperparameter Tuning using Grid Search
# test different kernels and C values to find the mathematical optimum
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.001],
    'kernel': ['linear', 'rbf']
}

print("Running Grid Search for SVM... (This may take a moment)")
svm_grid = GridSearchCV(SVC(probability=True), param_grid, cv=5, scoring='accuracy', verbose=1)
svm_grid.fit(X_train_scaled, y_train)

# Extract the best model
best_svm = svm_grid.best_estimator_
print(f"Best Parameters: {svm_grid.best_params_}")

# Predictions and detailed evaluation
svm_preds = best_svm.predict(X_test_scaled)
svm_probs = best_svm.predict_proba(X_test_scaled)[:, 1]

print("\n--- SVM Final Evaluation ---")
print(classification_report(y_test, svm_preds))

Running Grid Search for SVM... (This may take a moment)
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}

--- SVM Final Evaluation ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.99        71
           1       1.00      0.95      0.98        43

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

